In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
train_df = pd.read_csv("../data/clean/train_processed.csv")
test_df = pd.read_csv("../data/clean/test_processed.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (9975, 29)
Test shape: (2494, 29)


In [3]:
X_train = train_df.drop(columns=["FinalGrade"])
y_train = train_df["FinalGrade"]

X_test = test_df.drop(columns=["FinalGrade"])
y_test = test_df["FinalGrade"]

In [4]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (9975, 28)
y_train: (9975,)
X_test : (2494, 28)
y_test : (2494,)


In [5]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [6]:
y_pred_logistic = logistic_model.predict(X_test)

In [7]:
logistic_accuracy = accuracy_score(y_test, y_pred_logistic)

print("Logistic Regression Accuracy:", logistic_accuracy)
print(f"Accuracy: {logistic_accuracy * 100:.2f}%")

Logistic Regression Accuracy: 0.28869286287089013
Accuracy: 28.87%


In [8]:
print("Classification Report:")
print(classification_report(y_test, y_pred_logistic))

Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.52      0.39       680
           1       0.23      0.11      0.14       589
           2       0.29      0.28      0.28       644
           3       0.28      0.22      0.24       581

    accuracy                           0.29      2494
   macro avg       0.28      0.28      0.26      2494
weighted avg       0.28      0.29      0.27      2494



In [9]:
cm_logistic = confusion_matrix(y_test, y_pred_logistic)

print("Confusion Matrix:")
print(cm_logistic)

Confusion Matrix:
[[353  67 165  95]
 [268  62 137 122]
 [283  73 179 109]
 [247  65 143 126]]


In [10]:
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [11]:
y_pred_rf = random_forest_model.predict(X_test)

In [12]:
rf_accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_accuracy)
print(f"Accuracy: {rf_accuracy * 100:.2f}%")

Random Forest Accuracy: 0.888532477947073
Accuracy: 88.85%


In [13]:
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.91      0.90       680
           1       0.88      0.85      0.86       589
           2       0.88      0.89      0.89       644
           3       0.91      0.90      0.90       581

    accuracy                           0.89      2494
   macro avg       0.89      0.89      0.89      2494
weighted avg       0.89      0.89      0.89      2494



In [14]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Random Forest Confusion Matrix:")
print(cm_rf)

Random Forest Confusion Matrix:
[[622  18  26  14]
 [ 37 499  30  23]
 [ 27  29 575  13]
 [ 20  20  21 520]]


In [15]:
from sklearn.model_selection import cross_val_score

In [16]:
cv_scores = cross_val_score(
    random_forest_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

print("Cross-validation scores:", cv_scores)
print(f"Mean CV Accuracy: {cv_scores.mean() * 100:.2f}%")
print(f"CV Standard Deviation: {cv_scores.std() * 100:.2f}%")

Cross-validation scores: [0.82857143 0.83057644 0.8085213  0.82907268 0.83358396]
Mean CV Accuracy: 82.61%
CV Standard Deviation: 0.89%


In [17]:
from sklearn.model_selection import RandomizedSearchCV

In [18]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [19]:
rf_random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

rf_random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20, 30],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300]},
                   random_state=42, scoring='accuracy')

In [20]:
print("Best Hyperparameters:")
print(rf_random_search.best_params_)

Best Hyperparameters:
{'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20}


In [21]:
print("Best CV Accuracy:")
print(f"{rf_random_search.best_score_ * 100:.2f}%")

Best CV Accuracy:
80.07%


In [22]:
best_rf_model = rf_random_search.best_estimator_

print("Best Model:")
print(best_rf_model)

Best Model:
RandomForestClassifier(max_depth=20, min_samples_split=5, n_jobs=-1,
                       random_state=42)


In [23]:
y_pred_best_rf = best_rf_model.predict(X_test)

In [24]:
best_rf_accuracy = accuracy_score(y_test, y_pred_best_rf)

print("Tuned Random Forest Accuracy:", best_rf_accuracy)
print(f"Accuracy: {best_rf_accuracy * 100:.2f}%")

Tuned Random Forest Accuracy: 0.867682437850842
Accuracy: 86.77%


In [25]:
print("Tuned Random Forest Classification Report:")
print(classification_report(y_test, y_pred_best_rf))

Tuned Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.90      0.88       680
           1       0.86      0.82      0.84       589
           2       0.86      0.88      0.87       644
           3       0.89      0.87      0.88       581

    accuracy                           0.87      2494
   macro avg       0.87      0.87      0.87      2494
weighted avg       0.87      0.87      0.87      2494



In [26]:
feature_importance = pd.Series(
    random_forest_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(feature_importance)

num__AssignmentCompletion    0.144440
num__Attendance              0.139260
num__StudyHours              0.128770
num__OnlineCourses           0.128369
num__Age                     0.109603
cat__StressLevel_2           0.019928
cat__StressLevel_1           0.018300
cat__LearningStyle_3         0.018093
cat__EduTech_1               0.017874
cat__EduTech_0               0.017770
cat__LearningStyle_0         0.017255
cat__LearningStyle_2         0.017120
cat__Gender_1                0.016847
cat__Gender_0                0.016754
cat__LearningStyle_1         0.016332
cat__Resources_1             0.016051
cat__Discussions_1           0.015978
cat__Discussions_0           0.015864
cat__Motivation_1            0.015392
cat__Resources_2             0.014839
cat__Extracurricular_1       0.014330
cat__Extracurricular_0       0.014120
cat__Motivation_0            0.013965
cat__StressLevel_0           0.013277
cat__Motivation_2            0.013002
cat__Resources_0             0.012777
cat__Interne

In [27]:
print("\nTop 10 Features:")
print(feature_importance.head(10))


Top 10 Features:
num__AssignmentCompletion    0.144440
num__Attendance              0.139260
num__StudyHours              0.128770
num__OnlineCourses           0.128369
num__Age                     0.109603
cat__StressLevel_2           0.019928
cat__StressLevel_1           0.018300
cat__LearningStyle_3         0.018093
cat__EduTech_1               0.017874
cat__EduTech_0               0.017770
dtype: float64


In [29]:
import joblib

joblib.dump(
    random_forest_model,
    "../models/random_forest_model.joblib"
)

print("Random Forest model saved successfully.")

Random Forest model saved successfully.


In [35]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "Test Accuracy": [
        logistic_accuracy,
        rf_accuracy,
        best_rf_accuracy
    ],
    "CV Accuracy": [
        np.nan,
        cv_scores.mean(),
        rf_random_search.best_score_
    ]
})

model_comparison["Test Accuracy"] = (
    model_comparison["Test Accuracy"] * 100
).round(2)

model_comparison["CV Accuracy"] = (
    model_comparison["CV Accuracy"] * 100
).round(2)

model_comparison

,Model,Test Accuracy,CV Accuracy
0,Logistic Regression,28.87,NaN
1,Random Forest,88.85,82.61
2,Tuned Random Forest,86.77,80.07


In [36]:
final_model = random_forest_model

print("Final Model: Random Forest Classifier")
print(f"Test Accuracy: {rf_accuracy * 100:.2f}%")
print(f"CV Accuracy: {cv_scores.mean() * 100:.2f}%")

Final Model: Random Forest Classifier
Test Accuracy: 88.85%
CV Accuracy: 82.61%
